# DeepSVDD baseline on MSL

This notebook trains a standalone DeepSVDD baseline with EasyTSAD and reports Precision, Recall, F1, and AUPRC.

In [1]:
# Cell 1 — Setup

!pip install --upgrade pip
!pip install toml torch torchinfo optuna scikit-learn
!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git

# Clone repos
!rm -rf KAN-AD datasets
!git clone https://github.com/CSTCloudOps/KAN-AD.git
!git clone https://github.com/CSTCloudOps/datasets.git
!mv datasets KAN-AD/datasets

%cd KAN-AD

# Path setup
import sys, os, random
import numpy as np
import torch

project_path = os.getcwd()
if project_path not in sys.path:
    sys.path.append(project_path)

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

print("🟢 Paths configured.")
print("Current working directory:", project_path)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# Fix EasyTSAD import bug
!sed -i 's/TSData、/TSData/g; s/TSData,*/TSData/g' \
    /usr/local/lib/python3.12/dist-packages/EasyTSAD/DataFactory/__init__.py

from EasyTSAD.Controller import TSADController
from EasyTSAD.Methods import BaseMethod
print("✅ EasyTSAD ready")


  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-mp1e3zte
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-mp1e3zte
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit 507e5533a142eec7eece4372c55e83bb3faff8a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 2.57 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-reu

In [2]:
# Cell 2 — Dataset selection

DATASET = "MSL"
DISPLAY_MODEL = "DeepSVDD"
METHOD_NAME = "DeepSVDD_TSAD"
TRAINING_SCHEMA = "naive"

print("Dataset:", DATASET)


Dataset: MSL


In [3]:
# Cell 3 — DeepSVDD model and EasyTSAD wrapper

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import tqdm
from torch.utils.data import Dataset, DataLoader
from EasyTSAD.Methods import BaseMethod

def clean_array(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(x, -1e6, 1e6).astype(np.float32)

class MTSWindowDataset(Dataset):
    def __init__(self, tsData, phase, window_size):
        self.window_size = int(window_size)
        if phase == "train":
            self.data = clean_array(tsData.train)
        elif phase == "valid":
            self.data = clean_array(tsData.valid)
        elif phase == "test":
            self.data = clean_array(tsData.test)
        else:
            raise ValueError("phase must be train / valid / test")
        assert self.data.ndim == 2, f"Expected 2D MTS array, got {self.data.shape}"
        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        return torch.tensor(x, dtype=torch.float32)

class DeepSVDDEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256, bias=False),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.1),
            nn.Linear(256, 128, bias=False),
            nn.LayerNorm(128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, emb_dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)

class DeepSVDD_TSAD(BaseMethod):
    def __init__(self, params: dict) -> None:
        super().__init__()
        self.__anomaly_score = None
        self.window = int(params["window"])
        self.batch_size = int(params["batch_size"])
        self.epochs = int(params["epochs"])
        self.lr = float(params["lr"])
        self.patience = int(params.get("patience", 5))
        self.emb_dim = int(params.get("emb_dim", 64))
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.optimizer = None
        self.center = None
        self.best_state = None

    def _init_model(self, feature_dim):
        input_dim = self.window * feature_dim
        self.model = DeepSVDDEncoder(input_dim=input_dim, emb_dim=self.emb_dim).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-6)

    def _embed(self, x):
        # x: (B, W, F)
        B = x.shape[0]
        x_flat = x.reshape(B, -1)
        z = self.model(x_flat)
        z = torch.nan_to_num(z, nan=0.0, posinf=1e3, neginf=-1e3)
        z = torch.clamp(z, -50.0, 50.0)
        return z

    def _compute_center(self, loader):
        self.model.eval()
        zs = []
        with torch.no_grad():
            for x in tqdm.tqdm(loader, desc="Compute SVDD center"):
                x = x.to(self.device)
                z = self._embed(x)
                zs.append(z.detach().cpu())
        Z = torch.cat(zs, dim=0)
        c = Z.mean(dim=0)
        c[(c.abs() < 1e-6)] = 1e-6
        self.center = c.to(self.device)

    def _distance(self, x):
        z = self._embed(x)
        dist = ((z - self.center) ** 2).sum(dim=1)
        dist = torch.nan_to_num(dist, nan=1e6, posinf=1e6, neginf=1e6)
        dist = torch.clamp(dist, 0.0, 1e6)
        return dist

    def train_valid_phase(self, tsTrain):
        train_ds = MTSWindowDataset(tsTrain, "train", self.window)
        valid_ds = MTSWindowDataset(tsTrain, "valid", self.window)

        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True, drop_last=False)
        valid_loader = DataLoader(valid_ds, batch_size=self.batch_size, shuffle=False, drop_last=False)

        sample_x = next(iter(train_loader))
        feature_dim = sample_x.shape[-1]
        self._init_model(feature_dim)

        self._compute_center(train_loader)

        best_valid = float("inf")
        patience_counter = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            train_losses = []
            for x in tqdm.tqdm(train_loader, desc=f"Train {epoch}"):
                x = x.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                dist = self._distance(x)
                loss = dist.mean()
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
                self.optimizer.step()
                train_losses.append(float(loss.item()))

            self.model.eval()
            valid_losses = []
            with torch.no_grad():
                for x in tqdm.tqdm(valid_loader, desc=f"Valid {epoch}"):
                    x = x.to(self.device)
                    dist = self._distance(x)
                    loss = dist.mean()
                    if torch.isfinite(loss):
                        valid_losses.append(float(loss.item()))

            train_loss = np.mean(train_losses) if train_losses else np.inf
            valid_loss = np.mean(valid_losses) if valid_losses else np.inf
            print(f"Epoch {epoch} | train_loss={train_loss:.6f} | valid_loss={valid_loss:.6f}")

            if np.isfinite(valid_loss) and valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0
                self.best_state = {
                    "model": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                    "center": self.center.detach().cpu().clone()
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        if self.best_state is not None:
            self.model.load_state_dict(self.best_state["model"])
            self.center = self.best_state["center"].to(self.device)

    def test_phase(self, tsData):
        test_loader = DataLoader(
            MTSWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )
        self.model.eval()
        scores = []
        with torch.no_grad():
            for x in test_loader:
                x = x.to(self.device)
                dist = self._distance(x)
                scores.append(dist.cpu().numpy())

        scores = np.concatenate(scores) if scores else np.array([])
        if len(scores) == 0:
            padded = np.zeros(len(tsData.test))
        else:
            prefix = np.full(self.window, scores[0])
            padded = np.concatenate([prefix, scores])[:len(tsData.test)]

        self.__anomaly_score = padded.astype(np.float64)

    def anomaly_score(self) -> np.ndarray:
        return self.__anomaly_score

print("✅ DeepSVDD EasyTSAD method ready")


✅ DeepSVDD EasyTSAD method ready


In [4]:
# Cell 4 — Create config file

import os

config_text = """\
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
batch_size = 256
epochs = 30
lr = 0.001
patience = 5
emb_dim = 64
"""

CFG_PATH = os.path.join("kanad", f"config_deepsvdd_{DATASET.lower()}.toml")

with open(CFG_PATH, "w") as f:
    f.write(config_text)

print("✅ Config written to:", CFG_PATH)
print(open(CFG_PATH).read())


✅ Config written to: kanad/config_deepsvdd_msl.toml
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
batch_size = 256
epochs = 30
lr = 0.001
patience = 5
emb_dim = 64



In [5]:
# Cell 5 — Train DeepSVDD

gctrl = TSADController()

gctrl.set_dataset(
    dataset_type="MTS",
    dirname="datasets",
    datasets=[DATASET],
)

print(f"🚀 Training DeepSVDD on {DATASET}...")

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path=CFG_PATH,
)

print("✅ Training finished")


(2026-06-05 22:04:45,964) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

🚀 Training DeepSVDD on MSL...


(2026-06-05 22:04:46,339) [INFO]:     [DeepSVDD_TSAD] handling dataset MSL | curve AllInOne 
INFO:logger:    [DeepSVDD_TSAD] handling dataset MSL | curve AllInOne 
Valid 1: 100%|██████████| 228/228 [00:01<00:00, 192.67it/s]


Epoch 1 | train_loss=0.126223 | valid_loss=0.003663


Valid 2: 100%|██████████| 228/228 [00:01<00:00, 131.35it/s]


...

Epoch 20 | train_loss=0.000213 | valid_loss=0.000162


Valid 21: 100%|██████████| 228/228 [00:01<00:00, 187.88it/s]


Epoch 21 | train_loss=0.000209 | valid_loss=0.000161


Valid 22: 100%|██████████| 228/228 [00:01<00:00, 157.96it/s]


Epoch 22 | train_loss=0.000203 | valid_loss=0.000127
Early stopping
✅ Training finished


In [6]:
# Cell 6 — Evaluate DeepSVDD on MSL

from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

method = METHOD_NAME
training_schema = TRAINING_SCHEMA

gctrl.set_evals([
    PointF1PA(),
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(method=method, training_schema=training_schema)

print("✅ Evaluation completed")


(2026-06-05 22:06:24,911) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-06-05 22:06:24,913) [INFO]: Perform evaluations. Method[DeepSVDD_TSAD], Schema[naive].
INFO:logger:Perform evaluations. Method[DeepSVDD_TSAD], Schema[naive].
(2026-06-05 22:06:24,917) [INFO]:     [Load Data (All)] DataSets: MSL 
INFO:logger:    [Load Data (All)] DataSets: MSL 
(2026-06-05 22:06:24,949) [INFO]:     [DeepSVDD_TSAD] Eval dataset MSL <<<
INFO:logger:    [DeepSVDD_TSAD] Eval dataset MSL <<<
(2026-06-05 22:06:24,951) [INFO]:         [MSL] Using default margins (0, 5)
INFO:logger:        [MSL] Using default margins (0, 5)


✅ Evaluation completed


In [7]:
# Cell 7 — Load results and convert to comparison row

import json, os

avg_path = f"/content/KAN-AD/Results/Evals/{METHOD_NAME}/{TRAINING_SCHEMA}/{DATASET}/avg.json"
all_path = f"/content/KAN-AD/Results/Evals/{METHOD_NAME}/{TRAINING_SCHEMA}/{DATASET}/all.json"

assert os.path.exists(avg_path), f"avg.json not found: {avg_path}"
assert os.path.exists(all_path), f"all.json not found: {all_path}"

with open(avg_path, "r") as f:
    avg = json.load(f)

with open(all_path, "r") as f:
    all_scores = json.load(f)

print(f"=== AVERAGE RESULTS ({DISPLAY_MODEL} - {DATASET}) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

print("\n=== PER-SERIES RESULTS ===")
print("Number of series:", len(all_scores))
print("Example entry:")
print(list(all_scores.items())[0])

row = {
    "Model": DISPLAY_MODEL,
    "Dataset": DATASET,
    "F1": avg["best f1 under pa"]["f1"],
    "Precision": avg["best f1 under pa"]["precision"],
    "Recall": avg["best f1 under pa"]["recall"],
    "Event_F1": avg["event-based f1 under pa with mode squeeze"]["f1"],
    "Delay_F1": avg["best f1 under 5-delay pa"]["f1"],
    "AUPRC": avg["point-based auprc pa"],
}

print("\n=== Clean row for comparison table ===")
for k, v in row.items():
    print(f"{k}: {v}")


=== AVERAGE RESULTS (DeepSVDD - MSL) ===
best f1 under pa: {'f1': 0.7656639765223766, 'precision': 0.9180154820548909, 'recall': 0.6566826076013088, 'threshold': 0.014329268895380665}
event-based f1 under pa with mode squeeze: {'f1': 0.2105263157894732, 'precision': 0.2, 'recall': 0.2222222222222222, 'threshold': 0.018755703531496692}
best f1 under 5-delay pa: {'f1': 0.3037901086522035, 'precision': 0.2135594952230113, 'recall': 0.5260508431915429, 'threshold': 0.007644461300515104}
point-based auprc pa: 0.7761114268364264

=== PER-SERIES RESULTS ===
Number of series: 1
Example entry:
('AllInOne', {'best f1 under pa': {'f1': 0.7656639765223766, 'precision': 0.9180154820548909, 'recall': 0.6566826076013088, 'threshold': 0.014329268895380665}, 'event-based f1 under pa with mode squeeze': {'f1': 0.2105263157894732, 'precision': 0.2, 'recall': 0.2222222222222222, 'threshold': 0.018755703531496692}, 'best f1 under 5-delay pa': {'f1': 0.3037901086522035, 'precision': 0.2135594952230113, 'rec